# Retail demand forecasting — walkthrough

Every cell calls the same library functions the CLI, Prefect flows and tests use (`retailfc.panel`, `retailfc.features`, `retailfc.eda`, `retailfc.experiments`, `retailfc.report`). Set `RETAILFC_DATA_DIR` / `RETAILFC_ARTIFACTS_DIR` and optionally `RETAILFC_DATASET_VERSION` before starting Jupyter. **Results appear only if the experiments were run; otherwise sections say PENDING.**

In [ ]:
import os, json, pandas as pd
from pathlib import Path
from IPython.display import Markdown, Image, display
from retailfc import config as C, experiments as E, protocol as P, report
from retailfc.ingestion import raw
from retailfc.panel import load_panel_from_raw

cfg = C.load_config(os.environ.get("RETAILFC_CONFIG"))
version = os.environ.get("RETAILFC_DATASET_VERSION") or raw.latest_version()
manifest = raw.read_manifest(version)
print(version, manifest["observed_sales"])

## Dataset and information boundary
The manifest records source, checksums, schema version and observed date range.

In [ ]:
pd.DataFrame(manifest["files"]).T

In [ ]:
panel = load_panel_from_raw(version, max_series=int(os.environ.get("RETAILFC_NB_MAX_SERIES", 0)) or None)
origins = P.make_origins(panel.n_days, cfg)
print("series", panel.n_series, "days", panel.n_days, origins)

## Feature dictionary and leakage contract
Generated from `retailfc.features.spec` (single source of truth).

In [ ]:
from retailfc.features.spec import SPECS

pd.DataFrame([s.__dict__ for s in SPECS])[
    ["name", "group", "definition", "time_ref", "availability", "missing"]
]

## EDA (training period only)

In [ ]:
out = Path(os.environ.get("RETAILFC_ARTIFACTS_DIR", "artifacts")) / "eda_nb"
res = E_eda = __import__("retailfc.eda", fromlist=["x"]).run_eda(panel, cfg, out)
for f in res["findings"]:
    print("-", f)
for name in res["charts"].values():
    display(Image(str(out / name)))

## Experiment results (from stored outputs)

In [ ]:
exp_id = os.environ.get("RETAILFC_EXPERIMENT_ID") or E.make_experiment_id(cfg, version)
r = report.build_report(exp_id)
print(r["status"])
display(Markdown((Path(r["out"]) / "results.md").read_text(encoding="utf-8")))